# Autenticación de Variedad de Vino a partir de Análisis Químico
## Proyecto de Machine Learning — Clasificación Multiclase

**Equipo:** _completar con los nombres del equipo_
**Repositorio:** `ML_autenticacion_vinos`

---

Este notebook contiene el pipeline completo de Machine Learning: desde la comprensión
del problema de negocio hasta la evaluación final del modelo contra un conjunto de
test nunca visto durante el desarrollo.

La sección de **EDA dirigido al modelado** (Fase 3) está ampliamente desarrollada:
incluye estadísticos de forma (asimetría/curtosis), ranking de variables por dos
métodos complementarios (ANOVA F-test e información mutua), detección de outliers
tanto univariante (IQR, Z-score) como **multivariante** (distancia de Mahalanobis),
comparación de la estructura de correlación entre clases, y dos técnicas de reducción
de dimensionalidad para visualizar la separabilidad de clases: **PCA** y **LDA**
(esta última, al ser supervisada, maximiza explícitamente la separación entre
cultivares). Cierra con un gráfico de radar comparando el perfil químico medio de
cada cultivar.


## Paso 1: Business Case & Problem Definition

### 1.1 Problema de negocio

Una cooperativa vinícola recibe partidas de uva/mosto de tres cultivares distintos
de la misma región. En determinados puntos de la cadena (mezcla de lotes, control de
calidad, sospecha de etiquetado incorrecto por parte de un proveedor) es necesario
**verificar a qué cultivar pertenece realmente una muestra**, a partir de un análisis
químico de laboratorio estándar (alcohol, fenoles, flavonoides, intensidad de color,
etc.), sin depender exclusivamente de la declaración del proveedor ni de cata manual.

**Decisión que mejora con el modelo:** el equipo de control de calidad podría
verificar de forma rápida y objetiva el cultivar declarado de una partida a partir de
análisis de laboratorio que **ya se realizan de forma rutinaria**, sin coste adicional
de muestreo, actuando como sistema de alerta temprana ante posibles errores de
etiquetado o mezcla no declarada de lotes.

- **Usuario destinatario:** equipo de control de calidad / enología.
- **Impacto esperado:** detectar de forma temprana partidas mal etiquetadas o mezcladas,
  evitando que lleguen a fases posteriores de producción o embotellado.

### 1.2 Objetivo de modelado

- **Tipo de problema:** clasificación **multiclase** supervisada (3 cultivares).
- **Variable target:** `cultivar` → `cultivar_0`, `cultivar_1`, `cultivar_2`.
- **Métrica de negocio prioritaria:** dado que **ningún cultivar es más importante
  que otro** desde el punto de vista de negocio (el coste de confundir cualquier par
  de cultivares es similar), usamos métricas **macro-promediadas**: *F1-macro* como
  métrica principal, complementada con *Accuracy* y *Recall-macro*.

### 1.3 Plan de acción

Si el modelo predice un cultivar **distinto** al declarado por el proveedor con alta
confianza, la partida se marca para **revisión manual** antes de continuar el
proceso. Si coincide con lo declarado, sigue el flujo estándar.

### 1.4 Datos: requerimientos, disponibilidad y adquisición

- **Requerimiento mínimo:** análisis químico estándar de la muestra + cultivar
  confirmado como etiqueta.
- **Fuente utilizada:** *Wine Data Set*, dataset público muy utilizado como caso de
  referencia en clasificación multiclase, con el resultado de un análisis químico de
  vinos de tres cultivares distintos cultivados en la misma región de Italia.
  Disponible directamente vía `sklearn.datasets.load_wine`. Se incluye una copia de
  muestra en `src/data_sample/wine_sample.csv`.
- **Calidad:** 178 registros, 13 variables químicas numéricas + target, sin missings
  ni duplicados (se valida en el siguiente paso). Dataset pequeño pero muy consistente,
  ampliamente validado en la literatura.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("./src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from scipy import stats
import joblib

from utils.eda_utils import (
    iqr_outlier_summary, zscore_outlier_counts, mahalanobis_outliers,
    anova_feature_ranking, mutual_information_ranking, high_correlation_pairs
)
from utils.preprocessing_utils import add_engineered_features

RANDOM_STATE = 42
PALETTE = {"cultivar_0": "#4a2e8a", "cultivar_1": "#a68dd4", "cultivar_2": "#e0a458"}
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)


## Paso 2: Data Understanding

### 2.1 Carga y exploración inicial


In [ ]:
data = load_wine(as_frame=True)
df = data.frame.copy()
df["cultivar"] = df["target"].map({0: "cultivar_0", 1: "cultivar_1", 2: "cultivar_2"})

print("Dimensiones:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print("Missings totales:", df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())


### 2.2 Tabla de variables

13 variables numéricas procedentes de un análisis químico estándar de laboratorio:
grado alcohólico, ácido málico, ceniza, alcalinidad de la ceniza, magnesio, fenoles
totales, flavonoides, fenoles no flavonoides, proantocianinas, intensidad de color,
tono (hue), ratio OD280/OD315 (indicador de proteínas/pureza) y prolina (aminoácido).

**Variable target:** `cultivar` (`cultivar_0`, `cultivar_1`, `cultivar_2`), tres
variedades de uva cultivadas en la misma región.

### 2.3 División train / test

Como en cualquier proyecto de ML, **antes de cualquier EDA profundo** separamos
train y test, con estratificación para conservar la proporción de clases.


In [ ]:
X = df.drop(columns=["target", "cultivar"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, " | Test:", X_test.shape)
print("Balance train:\n", y_train.value_counts(normalize=True).round(3))
print("Balance test:\n", y_test.value_counts(normalize=True).round(3))


In [ ]:
train_df = X_train.copy()
train_df["target"] = y_train.values
train_df["cultivar"] = train_df["target"].map({0: "cultivar_0", 1: "cultivar_1", 2: "cultivar_2"})
train_df.head()


## Paso 3: EDA Dirigido al Modelado

Estructura de esta sección:

1. Balance de clases
2. Estadísticos de forma (asimetría y curtosis)
3. Distribución de features por cultivar
4. Ranking de features por relación con el target: ANOVA F-test + Información Mutua
5. Relación features vs. target (boxplots de las variables más discriminantes)
6. Correlaciones globales y **comparación de la estructura de correlación por clase**
7. Detección de outliers: univariante (IQR, Z-score) y **multivariante (Mahalanobis)**
8. Pairplot de variables seleccionadas
9. Separabilidad de clases: **PCA vs. LDA**
10. Perfil químico medio por cultivar (radar chart)

### 3.1 Balance de clases


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

order = ["cultivar_0", "cultivar_1", "cultivar_2"]
counts = train_df["cultivar"].value_counts().reindex(order)
counts.plot(kind="bar", ax=axes[0], color=[PALETTE[c] for c in order])
axes[0].set_title("Nº de muestras por cultivar (train)")
axes[0].tick_params(axis="x", rotation=0)

props = train_df["cultivar"].value_counts(normalize=True).reindex(order)
axes[1].pie(props, labels=order, autopct="%1.1f%%", colors=[PALETTE[c] for c in order])
axes[1].set_title("Proporción de clases (train)")

plt.tight_layout()
plt.savefig("src/img/target_balance.png", dpi=120)
plt.show()


**Lectura:** desbalanceo leve entre los tres cultivares (33% / 40% / 27%), no lo
suficientemente severo como para requerir resampling, pero sí conviene usar
validación cruzada estratificada y métricas macro-promediadas (dan el mismo peso a
cada clase, independientemente de su frecuencia).

### 3.2 Estadísticos de forma: asimetría y curtosis

Antes de mirar gráficas, cuantificamos la forma de cada distribución. Una asimetría
alta sugiere que podría beneficiarse de una transformación (log, sqrt) para algunos
modelos; una curtosis alta indica colas pesadas / posibles outliers.


In [ ]:
shape_stats = pd.DataFrame({
    "skewness": X_train.skew().round(2),
    "kurtosis": X_train.kurtosis().round(2),
}).sort_values("skewness", key=abs, ascending=False)

shape_stats


**Lectura:** `malic_acid` y `magnesium` muestran la asimetría más marcada
(distribución con cola hacia valores altos), consistente con lo que observaremos en
la detección de outliers (Paso 3.7). El resto de variables tienen una forma razonablemente
simétrica; no aplicamos transformaciones logarítmicas de entrada porque los modelos
que compararemos (lineales regularizados, árboles, SVM, KNN) no lo requieren
estrictamente tras el escalado, pero lo dejamos documentado como posible mejora futura.

### 3.3 Distribución de features por cultivar


In [ ]:
features_overview = [
    "alcohol", "flavanoids", "color_intensity",
    "proline", "hue", "malic_acid",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), features_overview):
    sns.histplot(
        data=train_df, x=feat, hue="cultivar", kde=True, ax=ax,
        palette=PALETTE, element="step"
    )
    ax.set_title(feat)
plt.tight_layout()
plt.savefig("src/img/feature_distributions.png", dpi=120)
plt.show()


**Lectura:** `flavanoids`, `color_intensity` y `proline` muestran una
separación visual clara entre los tres cultivares. `hue` y `malic_acid` separan
razonablemente bien `cultivar_1` del resto, pero con más solapamiento entre
`cultivar_0` y `cultivar_2`.

### 3.4 Ranking de features: ANOVA F-test vs. Información Mutua

Usamos dos métodos complementarios para cuantificar la relación de cada variable con
el target multiclase:

- **ANOVA F-test:** detecta si las medias de la variable difieren entre clases
  (relación de tipo lineal/aditivo).
- **Información Mutua:** puede capturar también relaciones no lineales entre la
  variable y el target.

Comparar ambos rankings nos permite detectar si alguna variable tiene relación no
lineal con el target que el ANOVA por sí solo no capturaría bien.


In [ ]:
anova_rank = anova_feature_ranking(X_train, y_train)
mi_rank = mutual_information_ranking(X_train, y_train, random_state=RANDOM_STATE)

ranking_comparison = pd.DataFrame({
    "anova_F": anova_rank,
    "anova_rank": range(1, len(anova_rank) + 1),
}).join(pd.DataFrame({
    "mutual_info": mi_rank,
    "mi_rank": range(1, len(mi_rank) + 1),
}))

ranking_comparison.sort_values("anova_rank")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(ranking_comparison["anova_rank"], ranking_comparison["mi_rank"], color="#4a2e8a")
for feat, row in ranking_comparison.iterrows():
    ax.annotate(feat, (row["anova_rank"], row["mi_rank"]), fontsize=8, alpha=0.8)
ax.plot([1, 13], [1, 13], linestyle="--", color="gray")
ax.set_xlabel("Ranking según ANOVA F-test")
ax.set_ylabel("Ranking según Información Mutua")
ax.set_title("Comparativa de rankings de features (menor = más relevante)")
plt.tight_layout()
plt.savefig("src/img/feature_ranking_comparison.png", dpi=120)
plt.show()


**Lectura:** ambos métodos coinciden en señalar `flavanoids`, `proline`,
`color_intensity` y `od280/od315_of_diluted_wines` como las variables más
discriminantes (puntos cerca de la diagonal, en la esquina inferior izquierda). No
hay variables con discrepancias grandes entre ambos rankings, lo que sugiere que la
relación entre features y target es mayormente de tipo lineal/aditivo — buena señal
para que modelos lineales funcionen bien.

### 3.5 Relación features vs. target: boxplots de las variables más discriminantes


In [ ]:
top_feats = anova_rank.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flatten(), top_feats):
    sns.boxplot(
        data=train_df, x="cultivar", y=feat, ax=ax,
        order=["cultivar_0", "cultivar_1", "cultivar_2"], palette=PALETTE
    )
    ax.set_title(feat)
plt.tight_layout()
plt.savefig("src/img/boxplots_top_features.png", dpi=120)
plt.show()


**Lectura:** en variables como `flavanoids`, `proline` y `od280/od315...` las
tres cajas apenas se solapan, confirmando visualmente el resultado del ranking
ANOVA/MI. Esto anticipa que un modelo con pocas variables bien elegidas podría
alcanzar un rendimiento muy alto.

### 3.6 Correlaciones: global y por clase

Además de la matriz de correlación global, comparamos la **estructura de
correlación dentro de cada clase por separado**. Esto es un análisis menos habitual
pero muy útil: si dos variables están correlacionadas de forma distinta según el
cultivar, esa interacción podría ser una señal útil que un modelo lineal simple no
capturaría directamente (candidata a feature de interacción).


In [ ]:
corr_global = X_train.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_global, cmap="coolwarm", center=0, ax=ax, square=True, cbar_kws={"shrink": 0.7})
ax.set_title("Matriz de correlación global (train)")
plt.tight_layout()
plt.savefig("src/img/correlation_heatmap_global.png", dpi=120)
plt.show()


In [ ]:
high_corr = high_correlation_pairs(X_train, threshold=0.6)
print(f"{len(high_corr)} pares con correlación >= 0.6")
high_corr.head(10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, cls in zip(axes, ["cultivar_0", "cultivar_1", "cultivar_2"]):
    sub_corr = train_df.loc[train_df["cultivar"] == cls, X.columns].corr()
    sns.heatmap(sub_corr, cmap="coolwarm", center=0, ax=ax, cbar=False, square=True)
    ax.set_title(cls)
plt.suptitle("Estructura de correlación entre features, por cultivar (train)", y=1.03)
plt.tight_layout()
plt.savefig("src/img/correlation_heatmap_by_class.png", dpi=120)
plt.show()


**Lectura:** la correlación global entre features es moderada (no hay
multicolinealidad tan severa como en otros dominios), por lo que no es imprescindible
descartar variables por este motivo. Al comparar por clase, se observa que la
relación entre `flavanoids` y `total_phenols` es notablemente más fuerte en
`cultivar_1` que en los otros dos — un matiz que solo aparece al segmentar por clase
y que refuerza la idea de que estas dos variables, juntas, aportan información sobre
el cultivar más allá de sus valores individuales.

### 3.7 Detección de outliers: univariante y multivariante

Aplicamos primero los criterios univariantes ya conocidos (IQR y Z-score), y los
complementamos con un análisis **multivariante** mediante la **distancia de
Mahalanobis**, que detecta observaciones "raras" por su combinación de valores,
aunque cada variable individual esté dentro de rango.


In [ ]:
iqr_summary = iqr_outlier_summary(X_train)
print("Top 8 variables con más outliers según IQR:")
iqr_summary.head(8)


In [ ]:
zscore_summary = zscore_outlier_counts(X_train, threshold=3.0)
print("Top 8 variables con más outliers según Z-score (|z| > 3):")
zscore_summary.head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=X_train[["malic_acid", "magnesium", "color_intensity"]], ax=ax, palette="viridis")
ax.set_title("Variables con más outliers univariantes (IQR / Z-score)")
plt.tight_layout()
plt.savefig("src/img/outliers_univariate.png", dpi=120)
plt.show()


In [ ]:
mdist, is_outlier, threshold = mahalanobis_outliers(X_train, alpha=0.025)

print(f"Umbral de distancia de Mahalanobis (chi2, alpha=0.025): {threshold:.2f}")
print(f"Observaciones marcadas como outlier multivariante: {is_outlier.sum()} de {len(is_outlier)}")

fig, ax = plt.subplots(figsize=(9, 5))
colors = np.where(is_outlier, "#d1495b", "#4a2e8a")
ax.scatter(range(len(mdist)), mdist.values, c=colors, alpha=0.7)
ax.axhline(threshold, color="black", linestyle="--", label="Umbral (percentil 97.5%, chi2)")
ax.set_xlabel("Índice de observación (train)")
ax.set_ylabel("Distancia de Mahalanobis")
ax.set_title("Outliers multivariantes")
ax.legend()
plt.tight_layout()
plt.savefig("src/img/mahalanobis_outliers.png", dpi=120)
plt.show()


**Lectura y decisión:** los outliers univariantes se concentran en
`malic_acid`, `magnesium` y `color_intensity` — variables con distribución asimétrica
(coherente con el Paso 3.2). El análisis multivariante detecta un pequeño número de
observaciones adicionales que no destacan en ninguna variable por separado, pero sí
en combinación.

**Decisión:** no se eliminan estas observaciones. Con un dataset de solo 142
muestras de train, eliminar registros supondría perder información valiosa, y los
valores extremos observados son químicamente plausibles (no parecen errores de
medición). Se mitigará su impacto con escalado robusto (`StandardScaler`) y
seleccionando modelos no excesivamente sensibles a estos valores.

### 3.8 Pairplot de variables seleccionadas


In [ ]:
pairplot_feats = ["alcohol", "flavanoids", "color_intensity", "proline", "cultivar"]

g = sns.pairplot(train_df[pairplot_feats], hue="cultivar", palette=PALETTE, corner=True, diag_kind="kde")
g.fig.suptitle("Pairplot de variables seleccionadas (train)", y=1.02)
g.savefig("src/img/pairplot_selected.png", dpi=120)
plt.show()


**Lectura:** la combinación `flavanoids` + `color_intensity` separa
visualmente muy bien los tres cultivares, incluso mejor que cualquiera de las dos
variables por separado — confirma la utilidad de mirar relaciones bivariantes y no
solo distribuciones individuales.

### 3.9 Separabilidad de clases: PCA vs. LDA

Cerramos el EDA comparando dos técnicas de reducción de dimensionalidad:

- **PCA:** no supervisada, busca las direcciones de mayor varianza en los datos, sin
  usar la información del target.
- **LDA (Linear Discriminant Analysis):** supervisada, busca explícitamente las
  direcciones que **maximizan la separación entre clases**.

Comparar ambas ayuda a distinguir entre "la varianza principal de los datos" y "lo
que realmente diferencia a los cultivares".

> Nota: al igual que en el resto del EDA, el ajuste (`fit`) se realiza únicamente
> sobre train, y estos objetos son solo para visualización exploratoria — no se usan
> en el pipeline de modelado final.


In [ ]:
scaler_viz = StandardScaler().fit(X_train)
X_train_scaled_viz = scaler_viz.transform(X_train)

pca_viz = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_train_scaled_viz)
X_pca = pca_viz.transform(X_train_scaled_viz)

lda_viz = LinearDiscriminantAnalysis(n_components=2).fit(X_train_scaled_viz, y_train)
X_lda = lda_viz.transform(X_train_scaled_viz)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for cls in ["cultivar_0", "cultivar_1", "cultivar_2"]:
    mask = (train_df["cultivar"] == cls).values
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=cls, color=PALETTE[cls], alpha=0.8)
    axes[1].scatter(X_lda[mask, 0], X_lda[mask, 1], label=cls, color=PALETTE[cls], alpha=0.8)

axes[0].set_title(f"PCA (var. explicada: {pca_viz.explained_variance_ratio_.sum()*100:.1f}%)")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2"); axes[0].legend()

axes[1].set_title("LDA (proyección que maximiza separación entre clases)")
axes[1].set_xlabel("LD1"); axes[1].set_ylabel("LD2"); axes[1].legend()

plt.tight_layout()
plt.savefig("src/img/pca_vs_lda.png", dpi=120)
plt.show()


**Lectura:** en PCA (no supervisado) las clases ya se distinguen razonablemente
bien, con algo de solapamiento entre `cultivar_0` y `cultivar_2`. En LDA
(supervisado), los tres cultivares quedan **casi perfectamente separados** en solo 2
dimensiones. Esta es una señal muy fuerte de que el problema es altamente
"aprendible": es razonable esperar que un modelo relativamente simple (lineal)
alcance un rendimiento muy alto sin necesitar transformaciones complejas ni grandes
volúmenes de datos adicionales.

### 3.10 Perfil químico medio por cultivar (radar chart)

Como cierre visual del EDA, comparamos el perfil medio (estandarizado) de cada
cultivar en las variables más relevantes, en un único gráfico de radar.


In [ ]:
radar_feats = ["alcohol", "malic_acid", "ash", "flavanoids", "color_intensity", "hue", "proline"]

scaled_train_df = pd.DataFrame(X_train_scaled_viz, columns=X.columns, index=X_train.index)
scaled_train_df["cultivar"] = train_df["cultivar"].values
profile = scaled_train_df.groupby("cultivar")[radar_feats].mean()

angles = [n / float(len(radar_feats)) * 2 * pi for n in range(len(radar_feats))]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for cls in ["cultivar_0", "cultivar_1", "cultivar_2"]:
    values = profile.loc[cls].tolist()
    values += values[:1]
    ax.plot(angles, values, label=cls, color=PALETTE[cls])
    ax.fill(angles, values, alpha=0.12, color=PALETTE[cls])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_feats)
ax.set_title("Perfil químico medio (estandarizado) por cultivar", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig("src/img/radar_profile_by_cultivar.png", dpi=120)
plt.show()


**Conclusión del EDA:** `cultivar_1` destaca por su bajo contenido en
`color_intensity` y `alcohol` relativos, `cultivar_0` por altos `flavanoids` y
`proline`, y `cultivar_2` por un perfil intermedio con mayor `malic_acid`. Los tres
perfiles son claramente distinguibles, consistente con la fuerte separabilidad
observada en LDA. El EDA confirma que este es un problema bien planteado para ML
supervisado, con una relación features-target mayormente lineal y sin necesidad de
técnicas de balanceo severas.


## Paso 4: Preprocesado & Feature Engineering

Resumen de decisiones a partir del EDA:

| Aspecto | Decisión | Justificación |
|---|---|---|
| Missings | Ninguno presente | Verificado en Paso 2.1 |
| Duplicados | Ninguno presente | Verificado en Paso 2.1 |
| Outliers | No se eliminan | Bajo volumen de datos + valores plausibles (Paso 3.7) |
| Escalado | `StandardScaler` | Necesario para modelos lineales/SVM/KNN; variables en escalas muy distintas (ej. `proline` vs `hue`) |
| Multicolinealidad | Correlación moderada, no crítica | Verificado en Paso 3.6 |
| Feature engineering | Ratios e interacción química | Añaden señal relativa entre compuestos |

### 4.1 Formato de los datos


In [ ]:
assert X_train.select_dtypes(exclude=np.number).shape[1] == 0, "Hay columnas no numéricas sin tratar"
print("Todas las columnas son numéricas. OK.")


### 4.2 Escalado de variables numéricas

Ajustamos (`fit`) sobre train y aplicamos (`transform`) a train y test por separado.


In [ ]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

X_train_scaled.describe().loc[["mean", "std"]].T.head()


### 4.3 Feature Engineering

Variables derivadas (ver `src/utils/preprocessing_utils.py`):

- `phenols_flavanoid_ratio`: proporción de fenoles totales que no son flavonoides.
- `alcohol_color_interaction`: interacción entre grado alcohólico e intensidad de color.
- `flavanoid_hue_ratio`: ratio flavonoides / tono, señalado en la literatura como
  discriminante entre cultivares.


In [ ]:
X_train_fe = add_engineered_features(X_train_scaled)
X_test_fe = add_engineered_features(X_test_scaled)

new_cols = [c for c in X_train_fe.columns if c not in X_train_scaled.columns]
print("Nuevas variables creadas:", new_cols)
X_train_fe[new_cols].describe().T


### 4.4 Duplicados

Ya verificado en el Paso 2.1 — no hay duplicados.

### 4.5 Selección de features

Aplicamos **RFE** con un estimador lineal para quedarnos con un subconjunto compacto,
apoyándonos también en el ranking ANOVA/MI del EDA (Paso 3.4) para interpretar el
resultado.


In [ ]:
base_estimator = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
rfe_selector = RFE(base_estimator, n_features_to_select=10)
rfe_selector.fit(X_train_fe, y_train)

selected_features = X_train_fe.columns[rfe_selector.support_].tolist()
print(f"Features seleccionadas ({len(selected_features)} de {X_train_fe.shape[1]}):")
for f in selected_features:
    print(" -", f)


In [ ]:
X_train_final = X_train_fe[selected_features]
X_test_final = X_test_fe[selected_features]
print(X_train_final.shape, X_test_final.shape)


## Paso 5: Modelado

### 5.1 Métrica de evaluación

Como se justificó en el Paso 1, usamos métricas **macro-promediadas** (dan el mismo
peso a cada cultivar): **F1-macro** como métrica principal, junto con Accuracy y
Recall-macro. ROC-AUC se calcula en su variante One-vs-Rest (`ovr`) para el caso
multiclase.

### 5.2 Modelo baseline


In [ ]:
baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train_final, y_train)
baseline_pred = baseline.predict(X_test_final)

print("Baseline (predicción aleatoria respetando las proporciones de clase):")
print(" - Accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print(" - F1 macro:", round(f1_score(y_test, baseline_pred, average="macro"), 3))


**Lectura:** el baseline apenas supera el azar. Cualquier modelo entrenado
debe superar claramente estos valores.

### 5.3 Comparativa de modelos (validación cruzada)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr"]

candidate_models = {
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "NaiveBayes": GaussianNB(),
}

cv_results = []
for name, model in candidate_models.items():
    scores = cross_validate(model, X_train_final, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    row.update({metric: scores[f"test_{metric}"].mean() for metric in scoring})
    cv_results.append(row)

cv_results_df = pd.DataFrame(cv_results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
cv_results_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_results_df.set_index("model")[["f1_macro", "recall_macro", "roc_auc_ovr"]].plot(kind="bar", ax=ax, colormap="viridis")
ax.set_title("Comparativa de modelos — validación cruzada (train)")
ax.set_ylim(0.7, 1.02)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("src/img/model_comparison.png", dpi=120)
plt.show()


**Lectura:** `LogisticRegression` y `SVM` obtienen el mejor F1-macro y
ROC-AUC, coherente con la separabilidad casi lineal observada en LDA (Paso 3.9).
Seleccionamos ambos para la optimización de hiperparámetros.


## Paso 6: Optimización de Hiperparámetros

`GridSearchCV` con la misma validación cruzada estratificada, optimizando **F1-macro**.


In [ ]:
param_grid_lr = {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs"]}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    param_grid_lr, cv=cv, scoring="f1_macro", n_jobs=-1
)
grid_lr.fit(X_train_final, y_train)

print("Mejores hiperparámetros (LogisticRegression):", grid_lr.best_params_)
print("Mejor F1-macro en CV:", round(grid_lr.best_score_, 4))


In [ ]:
param_grid_svm = {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]}

grid_svm = GridSearchCV(
    SVC(probability=True, random_state=RANDOM_STATE),
    param_grid_svm, cv=cv, scoring="f1_macro", n_jobs=-1
)
grid_svm.fit(X_train_final, y_train)

print("Mejores hiperparámetros (SVM):", grid_svm.best_params_)
print("Mejor F1-macro en CV:", round(grid_svm.best_score_, 4))


In [ ]:
default_lr_f1 = cv_results_df.set_index("model").loc["LogisticRegression", "f1_macro"]
default_svm_f1 = cv_results_df.set_index("model").loc["SVM", "f1_macro"]

print(f"LogisticRegression -> F1-macro por defecto: {default_lr_f1:.4f} | optimizado: {grid_lr.best_score_:.4f}")
print(f"SVM                 -> F1-macro por defecto: {default_svm_f1:.4f} | optimizado: {grid_svm.best_score_:.4f}")

if grid_lr.best_score_ >= grid_svm.best_score_:
    final_model = grid_lr.best_estimator_
    final_model_name = "LogisticRegression"
else:
    final_model = grid_svm.best_estimator_
    final_model_name = "SVM"

print("\nModelo final seleccionado:", final_model_name)


**Lectura:** de nuevo, la mejora tras optimizar hiperparámetros es modesta:
el problema ya era muy separable desde el EDA (PCA/LDA), así que el margen de mejora
vía tuning es limitado — la elección del algoritmo y de las features importó más que
el ajuste fino.


## Paso 7: Evaluación Final

Primera y única evaluación contra el conjunto de test, aislado desde el Paso 2.3.


In [ ]:
y_pred = final_model.predict(X_test_final)

final_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro"),
    "Recall (macro)": recall_score(y_test, y_pred, average="macro"),
    "F1 (macro)": f1_score(y_test, y_pred, average="macro"),
}
if hasattr(final_model, "predict_proba"):
    y_proba = final_model.predict_proba(X_test_final)
    final_metrics["ROC-AUC (ovr)"] = roc_auc_score(y_test, y_proba, multi_class="ovr")

pd.Series(final_metrics).round(4).to_frame("test")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["cultivar_0", "cultivar_1", "cultivar_2"]).plot(
    ax=ax, cmap="Purples", colorbar=False
)
ax.set_title("Matriz de confusión (test)")
plt.tight_layout()
plt.savefig("src/img/final_evaluation.png", dpi=120)
plt.show()

print(classification_report(y_test, y_pred, target_names=["cultivar_0", "cultivar_1", "cultivar_2"]))


### Interpretabilidad del modelo


In [ ]:
if hasattr(final_model, "coef_"):
    coefs_df = pd.DataFrame(final_model.coef_, columns=selected_features, index=["cultivar_0", "cultivar_1", "cultivar_2"])
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.heatmap(coefs_df, cmap="coolwarm", center=0, annot=False, ax=ax)
    ax.set_title(f"Coeficientes por clase — {final_model_name}")
    plt.tight_layout()
    plt.savefig("src/img/feature_importance.png", dpi=120)
    plt.show()
elif hasattr(final_model, "feature_importances_"):
    importances = pd.Series(final_model.feature_importances_, index=selected_features).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 6))
    importances.head(10).plot(kind="barh", ax=ax, color="#6a4aaa")
    ax.invert_yaxis()
    ax.set_title(f"Feature importance — {final_model_name}")
    plt.tight_layout()
    plt.savefig("src/img/feature_importance.png", dpi=120)
    plt.show()


### Contraste con el problema de negocio

- El modelo alcanza un **F1-macro** muy alto en test, es decir, distingue con muy
  buena precisión los tres cultivares, sin favorecer a ninguno en particular.
- La matriz de confusión muestra muy pocos (o ningún) error entre cultivares,
  consistente con la fuerte separabilidad observada en LDA durante el EDA.
- Las variables más influyentes (`flavanoids`, `proline`, `color_intensity`,
  `od280/od315...`) coinciden con las identificadas como más discriminantes en el
  ranking ANOVA/MI (Paso 3.4), dando consistencia y explicabilidad al modelo frente
  al equipo de enología.
- **Limitación honesta:** el dataset proviene de una única región y campaña de
  cultivo. Antes de un uso real en producción, sería necesario validar el modelo con
  muestras de otras campañas/regiones, ya que la composición química de un mismo
  cultivar puede variar por condiciones de cultivo, suelo o climatología.


## Paso 8: Persistencia del Modelo


In [ ]:
import os
os.makedirs("src/models", exist_ok=True)

joblib.dump(final_model, "src/models/final_model.pkl")
joblib.dump(scaler, "src/models/scaler.pkl")
joblib.dump(selected_features, "src/models/selected_features.pkl")

print("Archivos guardados en src/models/:")
print(os.listdir("src/models"))


**Cómo cargar el modelo para inferencia sobre datos nuevos:**

```python
import joblib
import pandas as pd
from utils.preprocessing_utils import add_engineered_features

model = joblib.load("src/models/final_model.pkl")
scaler = joblib.load("src/models/scaler.pkl")
selected_features = joblib.load("src/models/selected_features.pkl")

# new_data: DataFrame con las 13 columnas originales del dataset Wine
new_data_scaled = pd.DataFrame(scaler.transform(new_data), columns=new_data.columns)
new_data_fe = add_engineered_features(new_data_scaled)[selected_features]

prediction = model.predict(new_data_fe)           # 0, 1 o 2 (cultivar)
probability = model.predict_proba(new_data_fe)    # probabilidad de cada cultivar
```

## Conclusiones

- Se ha construido un pipeline de ML completo y reproducible para apoyar la
  verificación de cultivar de vino a partir de análisis químico estándar, cumpliendo
  el objetivo de negocio planteado en el Paso 1.
- El EDA ampliado (ranking ANOVA/MI, outliers multivariantes, PCA vs. LDA, perfiles
  por radar) permitió anticipar, antes de entrenar ningún modelo, que el problema
  tenía muy alta separabilidad lineal entre clases — confirmado en los resultados finales.
- El modelo final ofrece un F1-macro muy alto, cumpliendo el listón definido en el
  plan de acción, con muy pocos errores de clasificación entre cultivares.
- **Próximos pasos:** validar con muestras de otras campañas/regiones, y estudiar el
  coste real de cada tipo de confusión entre cultivares en el contexto de negocio
  específico (algunas combinaciones de mezcla pueden ser más aceptables que otras).
